<a href="https://colab.research.google.com/github/bisu617/ai-misinformation-detection/blob/main/model-stage2/robertaa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers datasets accelerate evaluate scikit-learn

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:

import pandas as pd

# CHANGE THIS to your actual dataset path in Drive
DATASET_PATH = '/content/drive/MyDrive/merged_cleaned_misinfo.csv'

df = pd.read_csv(DATASET_PATH)

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('\nLabel distribution:')
print(df['final_int'].value_counts())
print('\nSample:')
print(df[['clean_text', 'final_int']].head(3))

In [ ]:

from sklearn.model_selection import train_test_split

# Use clean_text as input and final_int as label
df = df[['clean_text', 'final_int']].copy()
df = df.rename(columns={'clean_text': 'text', 'final_int': 'label'})

# Clean up
df['text']  = df['text'].astype(str).str.strip()
# Drop rows where 'label' is NaN before converting to int
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)
df = df[df['text'].str.len() >= 30]
df = df[df['label'].isin([0, 1])]
df = df.drop_duplicates(subset=['text'])
df = df.reset_index(drop=True)

print('Cleaned shape:', df.shape)
print('Label distribution:')
print(df['label'].value_counts())

# Split: 80% train, 10% val, 10% test
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df,  test_df  = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f'\nTrain: {train_df.shape}')
print(f'Val:   {val_df.shape}')
print(f'Test:  {test_df.shape}')

In [ ]:

from datasets import Dataset
from transformers import AutoTokenizer

model_name = 'roberta-base'
tokenizer  = AutoTokenizer.from_pretrained(model_name)

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True))

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=128, padding='max_length')

train_tok = train_ds.map(tokenize, batched=True, remove_columns=['text'])
val_tok   = val_ds.map(tokenize,   batched=True, remove_columns=['text'])
test_tok  = test_ds.map(tokenize,  batched=True, remove_columns=['text'])

train_tok.set_format('torch')
val_tok.set_format('torch')
test_tok.set_format('torch')

print('Tokenization done!')
print(train_tok)


In [ ]:

import numpy as np
import evaluate
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

id2label = {0: 'factual',        1: 'misinformation'}
label2id = {'factual': 0, 'misinformation': 1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

acc_metric = evaluate.load('accuracy')
f1_metric  = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': acc_metric.compute(predictions=preds, references=labels)['accuracy'],
        'f1':       f1_metric.compute(predictions=preds,  references=labels, average='binary')['f1']
    }

print('Model loaded!')

In [ ]:

args = TrainingArguments(
    output_dir='roberta_stage2',
    eval_strategy='steps',
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=False,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    report_to='none'
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
torch.cuda.empty_cache()

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:

train_result = trainer.train()
print('\nTraining complete!')
print(f"Total train time : {train_result.metrics['train_runtime']:.1f}s")
print(f"Train samples/sec: {train_result.metrics['train_samples_per_second']:.1f}")

In [ ]:

import matplotlib.pyplot as plt

logs = trainer.state.log_history

train_steps  = [x['step'] for x in logs if 'loss' in x and 'eval_loss' not in x]
train_losses = [x['loss'] for x in logs if 'loss' in x and 'eval_loss' not in x]
val_steps    = [x['step'] for x in logs if 'eval_loss' in x]
val_losses   = [x['eval_loss'] for x in logs if 'eval_loss' in x]

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_losses, label='Train Loss', color='blue')
plt.plot(val_steps,   val_losses,   label='Val Loss',   color='orange', marker='o')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.title('RoBERTa Stage 2 - Train Loss vs Val Loss (Overfitting Check)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/AI_Humanizer/overfitting_stage2.png', dpi=150)
plt.show()
print('Overfitting plot saved!')

In [ ]:

test_results = trainer.evaluate(test_tok)
print('FINAL TEST RESULTS (STAGE 2) ')
print(f"Accuracy : {test_results['eval_accuracy']:.4f}")
print(f"F1 Score : {test_results['eval_f1']:.4f}")
print(f"Eval Loss: {test_results['eval_loss']:.4f}")


In [ ]:

from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

preds_output = trainer.predict(test_tok)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

print('\n=== Classification Report ===')
print(classification_report(y_true, y_pred, target_names=['Factual', 'Misinformation']))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Factual', 'Misinformation'])
disp.plot(cmap='Blues')
plt.title('RoBERTa Stage 2 - Confusion Matrix')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/AI_Humanizer/confusion_matrix_stage2.png', dpi=150)
plt.show()
print('Confusion matrix saved!')

In [ ]:
# CELL 12:
save_path = '/content/drive/MyDrive/AI_Humanizer/roberta_stage2_model'
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print('Stage 2 model saved to:', save_path)